# Example 1: Simple Format - ALS PBMC Dataset

This notebook demonstrates compiling a traditional GEO dataset where each sample has a single expression file.

**Dataset**: GSE244263 (ALS vs Control PBMC samples)  
**Format**: One .txt.gz file per sample  
**Samples**: 3 samples (2 ALS, 1 Control)

The pipeline compiles raw data into a QC-filtered, normalized h5ad. Downstream analysis (PCA, UMAP, clustering) is shown as an optional follow-up.

In [ ]:
import sys
import os
sys.path.append('../../')  # Add repo to path

from anndata_compiler import GEOAnndataCompiler
import pandas as pd

## 1. Examine the Data Structure

In [ ]:
# Look at the files
data_dir = '../data/simple_format_ALS'
print("Files in data directory:")
for f in os.listdir(data_dir):
    print(f"  {f}")

# Look at metadata
metadata = pd.read_csv(f'{data_dir}/metadata.csv')
print(f"\nMetadata ({metadata.shape[0]} samples):")
print(metadata[['Sample_name', 'Sample_geo_accession', 'Disease_state', 'Sample_ID']])

## 2. Configure the Compiler

For simple format, we just need basic configuration. QC filtering is applied automatically:

In [ ]:
config = {
    'raw_data_dir': data_dir,
    'metadata_file': f'{data_dir}/metadata.csv',
    'output_file': './als_compiled_example.h5ad',
    'sample_id_column': 'Sample_ID',
    
    # Processing parameters
    'max_cells_per_sample': 500,  # Small for demo; use None for full data
    'target_sum': 1e4,
    'n_top_genes': 2000,
    'delimiter': 'whitespace',
    
    # QC filtering
    'min_genes': 200,
    'min_cells': 3,
    'max_mito_pct': 20.0,
    
    # Metadata filtering
    'metadata_columns': ['Disease_state', 'Source']
}

print("Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

## 3. Run the Compilation Pipeline

In [ ]:
# Initialize compiler
compiler = GEOAnndataCompiler(config)

# Run pipeline (compiles, QC filters, normalizes, detects HVGs)
adata = compiler.run_full_pipeline()

print(f"\nFinal dataset: {adata.n_obs} cells x {adata.n_vars} genes")
print(f"Layers: {list(adata.layers.keys())}")
print(f"Samples: {adata.obs['sample_id'].unique()}")
print(f"Disease states: {adata.obs['Disease_state'].unique()}")

## 4. Inspect the Compiled Data

In [ ]:
import scanpy as sc

print("AnnData structure:")
print(adata)
print(f"\nObservation columns: {list(adata.obs.columns)}")
print(f"Variable columns: {list(adata.var.columns)}")
print(f"Layers: {list(adata.layers.keys())}")
print(f"HVGs: {adata.var['highly_variable'].sum()}")

# QC columns available
qc_cols = [c for c in adata.obs.columns if 'counts' in c or 'mt' in c]
print(f"QC columns: {qc_cols}")

# Sample composition
sample_counts = adata.obs.groupby(['sample_id', 'Disease_state']).size().unstack(fill_value=0)
print(f"\nCells per sample:")
print(sample_counts)

In [ ]:
# QC visualization (post-filtering)
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

## 5. Optional: Downstream Analysis Preview

The compiled h5ad is the output of the pipeline. Everything below is interactive analysis
that you'd normally do in your own notebook, inspecting results at each step.
See [docs/downstream_analysis_guide.md](../../docs/downstream_analysis_guide.md) for full details.

In [ ]:
# PCA and scree plot
sc.tl.pca(adata, svd_solver='arpack', n_comps=30, use_highly_variable=True)
sc.pl.pca_variance_ratio(adata, n_pcs=30, log=True)

# Neighbors, UMAP, and clustering
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=20)
sc.tl.umap(adata)

# Leiden at multiple resolutions
for res in [0.3, 0.5, 1.0]:
    sc.tl.leiden(adata, resolution=res, key_added=f'leiden_{res}')

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sc.pl.umap(adata, color='leiden_0.5', ax=axes[0], show=False, frameon=False)
axes[0].set_title('Leiden (res=0.5)')
sc.pl.umap(adata, color='Disease_state', ax=axes[1], show=False, frameon=False)
axes[1].set_title('Disease State')
sc.pl.umap(adata, color='sample_id', ax=axes[2], show=False, frameon=False)
axes[2].set_title('Sample ID')
plt.tight_layout()
plt.show()